# Sources and refs

## Overview
`source()` and `ref()` are the two functions that make dbt's lineage graph possible. They replace hardcoded table names with tracked references that dbt uses to build the DAG (Directed Acyclic Graph) of dependencies.

**`ref()` — reference another dbt model:**
```sql
FROM {{ ref('stg_orders') }}
-- Compiles to: FROM analytics.staging.stg_orders  (in prod)
-- dbt knows stg_orders must be built before this model
```

**`source()` — reference a raw table not managed by dbt:**
```sql
FROM {{ source('jaffle_shop', 'raw_orders') }}
-- Compiles to: FROM raw.jaffle_shop.raw_orders
-- Declared in a _sources.yml file
```

**Why not hardcode table names?**
- Hardcoded names break when schemas change or switching dev/prod targets
- `ref()` resolves to the correct schema for the current target automatically
- dbt uses these references to build the DAG and enforce build order
- `dbt docs generate` uses them to produce the visual lineage graph

**The lineage DAG:**
```
raw_orders (source)
    └── stg_orders (view)
            └── int_order_payments_joined (ephemeral)
                    └── fct_orders (table)

raw_customers (source)
    └── stg_customers (view)
            └── dim_customers (table)
                    └── fct_orders (table)  ← same node, two parents
```

---

In [ ]:
sources_yml = """
# models/staging/_sources.yml
version: 2

sources:
  - name: jaffle_shop
    database: raw
    schema: jaffle_shop
    description: "Raw transactional data loaded by Fivetran from the production app"

    freshness:
      warn_after:  {count: 6,  period: hour}
      error_after: {count: 24, period: hour}
    loaded_at_field: _loaded_at

    tables:
      - name: raw_orders
        description: "One row per order placed in the app"
        columns:
          - name: id
            description: "Primary key"
            tests:
              - unique
              - not_null
          - name: status
            tests:
              - accepted_values:
                  values: [placed, shipped, completed, return_pending, returned]

      - name: raw_customers
        description: "One row per registered customer"
        columns:
          - name: id
            tests: [unique, not_null]

      - name: raw_payments
        description: "One row per payment attempt"
        columns:
          - name: id
            tests: [unique, not_null]
          - name: payment_method
            tests:
              - accepted_values:
                  values: "{{ var('payment_methods') }}"
"""
print("=== _sources.yml: declaring raw tables as dbt sources ===")
print(sources_yml)

In [ ]:
dev_refs = [
    ("{{ source('jaffle_shop', 'raw_orders') }}", "raw.jaffle_shop.raw_orders"),
    ("{{ ref('stg_orders') }}",                    "analytics.dbt_dev_samantha.stg_orders"),
    ("{{ ref('fct_orders') }}",                    "analytics.dbt_dev_samantha_finance.fct_orders"),
]
prod_refs = [
    ("{{ source('jaffle_shop', 'raw_orders') }}", "raw.jaffle_shop.raw_orders"),
    ("{{ ref('stg_orders') }}",                    "analytics.staging.stg_orders"),
    ("{{ ref('fct_orders') }}",                    "analytics.finance.fct_orders"),
]
print("=== How ref() and source() compile ===")
print("  In dev (target: dev, schema: dbt_dev_samantha)")
for jinja, compiled in dev_refs:
    print(f"    {jinja:<48s}  ->  {compiled}")
print()
print("  In prod (target: prod, schema: analytics)")
for jinja, compiled in prod_refs:
    print(f"    {jinja:<48s}  ->  {compiled}")
print()
print("Key insight: the same model SQL works in dev and prod unchanged.")
print("Only the target profile changes -- dbt resolves all names automatically.")

---
## DAG selection and source freshness

In [ ]:
selections = [
    ("dbt run -s stg_orders",           "Build only stg_orders"),
    ("dbt run -s +fct_orders",          "Build fct_orders and ALL upstream dependencies"),
    ("dbt run -s fct_orders+",          "Build fct_orders and ALL downstream dependents"),
    ("dbt run -s +fct_orders+",         "Build fct_orders, all upstream, and all downstream"),
    ("dbt run -s staging.*",            "Build all models in the staging/ directory"),
    ("dbt run -s tag:daily",            "Build all models tagged daily"),
    ("dbt run -s config.materialized:incremental", "Build only incremental models"),
    ("dbt run -s source:jaffle_shop+",  "All models downstream of the jaffle_shop source"),
    ("dbt run --exclude fct_orders",    "Build everything EXCEPT fct_orders"),
    ("dbt run -s 1+fct_orders",         "fct_orders and only its direct parents (1 level up)"),
    ("dbt run -s fct_orders+1",         "fct_orders and only its direct children (1 level down)"),
]
print("=== DAG selection syntax ===")
print(f"  {chr(10).join(f"{c:<52s}  {d}" for c,d in selections)}")
print()
freshness = """
  dbt source freshness
  # Queries the loaded_at_field for each declared source.
  # Warns/errors based on thresholds in _sources.yml.
  # Example output:
  #   WARN  jaffle_shop.raw_orders   Last loaded 7h 23m ago (warn threshold: 6h)
  #   PASS  jaffle_shop.raw_payments Last loaded 2h 01m ago
"""
print("=== dbt source freshness check ===")
print(freshness)

---
## Common Pitfalls

**1. Hardcoding schema names instead of using ref() or source()**
`FROM analytics.staging.stg_orders` works in production but fails in a developer's personal schema. Every hardcoded name is a bug waiting to happen on a schema migration or developer onboarding. Always use `ref()` and `source()`.

**2. Declaring sources with wrong database or schema fields**
If `database` or `schema` in `_sources.yml` does not match the actual table location, `source()` compiles to the wrong path and models fail at runtime. Verify with `dbt compile -s stg_orders` and inspect the output in `target/compiled/`.

**3. Using `ref()` to point at a source table directly**
`{ ref('raw_orders') }` looks for a dbt model named `raw_orders`. There is no such model -- use `{ source('jaffle_shop', 'raw_orders') }` for raw tables. Mixing the two creates phantom dependencies in the DAG and confuses lineage.

**4. Circular dependencies between models**
Model A refs Model B which refs Model A is a cycle -- dbt cannot determine build order and raises an error. The fix is architectural: extract the shared logic into a third model that both can ref without creating a cycle.

**5. Not running `dbt source freshness` in production pipelines**
Sources go stale when upstream ingestion pipelines fail. A mart built on 3-day-old source data gives wrong results silently. Add `dbt source freshness` as the first step in every production run and alert on error-level threshold breaches.

**6. Using `staging.*+` without understanding cascading build scope**
`dbt run -s staging.*` builds everything in staging. `dbt run -s staging.*+` builds staging AND every downstream dependent of every staging model -- potentially the entire project. Understand the `+` modifier's cascading scope before using it in time-sensitive pipelines.


---
*sql_methods_library - Samantha McGarrigle*